In [2]:
import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect("../../amazon_electronics.db")
df = pd.read_sql_query("SELECT * FROM products", conn)
conn.close()

df.head()


,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating_count,about_product,img_link,product_link
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,399.0,1.099,0.64,24269,High Compatibility : Compatible With iPhone 12...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,199.0,349.000,0.43,43994,"Compatible with all Type C enabled devices, be...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,199.0,1.899,0.90,7928,【 Fast Charger& Data Sync】-With built-in safet...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,329.0,699.000,0.53,94363,The boAt Deuce USB 300 2 in 1 cable is compati...,https://m.media-amazon.com/images/I/41V5FtEWPk...,https://www.amazon.in/Deuce-300-Resistant-Tang...
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,154.0,399.000,0.61,16905,[CHARGE & SYNC FUNCTION]- This cable comes wit...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Portronics-Konnect-POR-1...


In [3]:
df["category"] = df["category"].str.split(r"[|&]")
df["category"] = df["category"].apply(lambda x: list(set(x)))

from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
category_matrix = mlb.fit_transform(df["category"])

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
price_scaled = scaler.fit_transform(df[["discounted_price"]])

In [4]:
feature_matrix = np.hstack((category_matrix, price_scaled))

from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(feature_matrix)

In [10]:
def get_nearest_products(product_id, similarity_matrix, df, k):
    product_index_map = pd.Series(
        df.index,
        index=df["product_id"]
    ).to_dict()

    if product_id not in product_index_map:
        return {}

    index = product_index_map[product_id]

    similarity_scores = list(enumerate(similarity_matrix[index]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    top_products = similarity_scores[1:k+1]

    k_nearest = {
        df.iloc[i]["product_id"]: score
        for i, score in top_products
    }

    return k_nearest

In [11]:
get_nearest_products(
    "B09X79PP8F",
    similarity_matrix,
    df,
    5
)

{'B0711PVX6Z': np.float64(1.0),
 'B09YLXYP7Y': np.float64(1.0),
 'B09X79PP8F': np.float64(1.0),
 'B09YLX91QR': np.float64(1.0),
 'B08WRWPM22': np.float64(0.9999999580306885)}

In [13]:
rated_produnts = {'B0789LZTCJ': 4.2, 'B094JNXNPV': 3.5}

def get_recommendations(rated_products, n):

    candidates = {}

    for p, r in rated_products.items():

        k_nearest = get_nearest_products(p,similarity_matrix,df,n*2)

        for product, cos_sim in k_nearest.items():
            if product in rated_products:
                continue
            if product in candidates:
                candidates[product] += float(cos_sim)
            else:
                candidates[product] = float(cos_sim) * r
    
    recommendations = [
        product_id 
        for product_id, _ in sorted(candidates.items(), key=lambda x: x[1], reverse=True)[:n]
    ]

    return recommendations


get_recommendations(rated_produnts, 5)



['B077Z65HSD', 'B0974H97TJ', 'B08HDH26JX', 'B0981XSZJ7', 'B00GE55L22']